# 04 - Logged Data-Cleaning Pipeline

This learning notebook begins **Part 2, Task 1**. It converts the raw-data audit into a reproducible Pandas cleaning workflow with schema validation, specific exception handling, categorical standardisation, duplicate removal, monthly median imputation and meaningful logging.

## notebook objectives

1. Load a CSV with specific error handling.
2. Validate the schema before any transformation.
3. Configure and use a named Python logger.
4. Standardise categorical values.
5. Parse and validate date/time values.
6. Remove exact duplicates without deleting legitimate repeated timestamps.
7. Impute impossible values using month-specific medians and explicit loops.
8. Validate the cleaned result before saving it.

> This notebook does not save the final cleaned CSV. After you validate the decisions, the logic will be promoted into the required `pipeline.py` with console and file logging.

## 1. Import packages and configure notebook logging

The notebook is an entry point, so it may configure a handler. The final reusable modules will call `logging.getLogger(__name__)`, while only their entry-point script configures handlers.

In [21]:
import logging
import platform
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

LOGGER = logging.getLogger("capstone.learning.data_cleaning")
LOGGER.handlers.clear()
LOGGER.setLevel(logging.DEBUG)
LOGGER.propagate = False
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)
console_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
    )
)
LOGGER.addHandler(console_handler)

LOGGER.info(
    "Environment ready: Python %s, Pandas %s, NumPy %s",
    platform.python_version(),
    pd.__version__,
    np.__version__,
)

2026-11-14 22:57:01,784 | INFO | capstone.learning.data_cleaning | Environment ready: Python 3.11.9, Pandas 3.1.1, NumPy 2.4.6


## 2. Locate the raw CSV without a computer-specific path

In [22]:
CSV_FILENAME = "Metro_Interstate_Traffic_Volume.csv"
EXPECTED_COLUMNS = [
    "holiday",
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "weather_main",
    "weather_description",
    "date_time",
    "traffic_volume",
]

def find_project_root(start: Path, filename: str) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / filename).is_file():
            return candidate
    raise FileNotFoundError(f"Could not find {filename!r}.")

PROJECT_ROOT = find_project_root(Path.cwd(), CSV_FILENAME)
CSV_PATH = PROJECT_ROOT / CSV_FILENAME
LOGGER.info("Using raw source file %s", CSV_PATH)

2026-11-14 22:57:01,788 | INFO | capstone.learning.data_cleaning | Using raw source file /Users/husa/software/AI/nus course/Capstone Project/Metro_Interstate_Traffic_Volume.csv


## 3. Load the CSV and validate its schema first

`keep_default_na=False` preserves the holiday text `None` as a valid category. The loader handles file, encoding and parsing problems separately; it does not use a bare `except`.

In [23]:
def validate_schema(frame: pd.DataFrame) -> None:
    actual_columns = frame.columns.tolist()
    missing_columns = sorted(set(EXPECTED_COLUMNS) - set(actual_columns))
    unexpected_columns = sorted(set(actual_columns) - set(EXPECTED_COLUMNS))
    if missing_columns or unexpected_columns or actual_columns != EXPECTED_COLUMNS:
        raise ValueError(
            "Schema mismatch: "
            f"missing={missing_columns}, unexpected={unexpected_columns}, "
            f"order_matches={actual_columns == EXPECTED_COLUMNS}"
        )
    LOGGER.info("Schema validation passed for %d expected columns", len(EXPECTED_COLUMNS))

def load_raw_csv(csv_path: Path) -> pd.DataFrame:
    try:
        frame = pd.read_csv(csv_path, keep_default_na=False)
    except FileNotFoundError:
        LOGGER.error("Raw CSV was not found: %s", csv_path, exc_info=True)
        raise
    except UnicodeDecodeError:
        LOGGER.error("Raw CSV encoding could not be decoded", exc_info=True)
        raise
    except pd.errors.ParserError:
        LOGGER.error("Raw CSV could not be parsed", exc_info=True)
        raise

    LOGGER.info("Loaded raw CSV with %d rows and %d columns", *frame.shape)
    validate_schema(frame)
    return frame

raw_frame = load_raw_csv(CSV_PATH)
display(raw_frame.head(3))

2026-11-14 22:57:01,806 | INFO | capstone.learning.data_cleaning | Loaded raw CSV with 48204 rows and 9 columns
2026-11-14 22:57:01,806 | INFO | capstone.learning.data_cleaning | Schema validation passed for 9 expected columns


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767


## 4. Profile issues before changing data

This profile distinguishes true nulls, blank text, exact duplicates and repeated timestamps. Repeated timestamps are not automatically removed because they contain different weather descriptions.

In [24]:
text_columns = raw_frame.select_dtypes(include=["object", "string"]).columns
blank_counts = {
    column: int(raw_frame[column].astype(str).str.strip().eq("").sum())
    for column in text_columns
}
initial_profile = {
    "rows": len(raw_frame),
    "columns": len(raw_frame.columns),
    "null_values": int(raw_frame.isna().sum().sum()),
    "blank_text_values": sum(blank_counts.values()),
    "exact_duplicate_rows": int(raw_frame.duplicated().sum()),
    "unique_timestamps": int(raw_frame["date_time"].nunique()),
    "additional_timestamp_records": (
        len(raw_frame) - int(raw_frame["date_time"].nunique())
    ),
    "zero_kelvin_rows": int(raw_frame["temp"].eq(0).sum()),
    "rain_above_9000_rows": int(raw_frame["rain_1h"].gt(9000).sum()),
    "zero_traffic_rows": int(raw_frame["traffic_volume"].eq(0).sum()),
}

LOGGER.info("Initial data-quality profile completed")
display(pd.Series(initial_profile, name="value").to_frame())

2026-11-14 22:57:01,847 | INFO | capstone.learning.data_cleaning | Initial data-quality profile completed


,value
rows,48204
columns,9
null_values,0
blank_text_values,0
exact_duplicate_rows,17
unique_timestamps,40575
additional_timestamp_records,7629
zero_kelvin_rows,10
rain_above_9000_rows,1
zero_traffic_rows,2


## 5. Standardise categorical text

Whitespace is removed from all categorical fields. Detailed weather descriptions are lower-cased so `Sky is Clear` and `sky is clear` become one category. Values that change are logged at WARNING level.

In [25]:
cleaned_frame = raw_frame.copy(deep=True)
category_change_counts = {}

for column in ["holiday", "weather_main", "weather_description"]:
    before = cleaned_frame[column].copy()
    standardized = before.astype("string").str.strip()
    if column == "weather_description":
        standardized = standardized.str.lower()
    cleaned_frame[column] = standardized.astype(str)
    affected_rows = int(before.ne(cleaned_frame[column]).sum())
    category_change_counts[column] = affected_rows
    if affected_rows > 0:
        LOGGER.warning(
            "Standardized %d values in categorical column %s",
            affected_rows,
            column,
        )
    else:
        LOGGER.info("Categorical column %s required no changes", column)

print("Category changes:", category_change_counts)
print("Unique weather descriptions:", cleaned_frame["weather_description"].nunique())

2026-11-14 22:57:01,861 | INFO | capstone.learning.data_cleaning | Categorical column holiday required no changes


2026-11-14 22:57:01,867 | INFO | capstone.learning.data_cleaning | Categorical column weather_main required no changes
2026-11-14 22:57:01,875 | WARNING | capstone.learning.data_cleaning | Standardized 1730 values in categorical column weather_description


Category changes: {'holiday': 0, 'weather_main': 0, 'weather_description': 1730}
Unique weather descriptions: 37


## 6. Parse and validate date/time values

The explicit format prevents ambiguous interpretation. A malformed value stops the pipeline with a logged error.

In [26]:
try:
    cleaned_frame["date_time"] = pd.to_datetime(
        cleaned_frame["date_time"],
        format="%Y-%m-%d %H:%M:%S",
        errors="raise",
    )
except (TypeError, ValueError) as exc:
    LOGGER.error("Date/time parsing failed: %s", exc, exc_info=True)
    raise

LOGGER.info(
    "Parsed date_time successfully; range %s to %s",
    cleaned_frame["date_time"].min(),
    cleaned_frame["date_time"].max(),
)

2026-11-14 22:57:01,886 | INFO | capstone.learning.data_cleaning | Parsed date_time successfully; range 2012-10-02 09:00:00 to 2018-09-30 23:00:00


## 7. Remove exact duplicates, preserve repeated weather observations

Duplicate removal occurs after standardisation and date parsing, so semantically identical rows are detected consistently. Different weather observations at the same timestamp remain available for weather analysis.

In [27]:
duplicate_count = int(cleaned_frame.duplicated().sum())
if duplicate_count > 0:
    cleaned_frame = cleaned_frame.drop_duplicates().reset_index(drop=True)
    LOGGER.warning(
        "Dropped %d exact duplicate rows; %d rows remain",
        duplicate_count,
        len(cleaned_frame),
    )
else:
    LOGGER.info("No exact duplicate rows were found")

timestamp_counts = cleaned_frame["date_time"].value_counts()
repeated_timestamp_rows = int(timestamp_counts[timestamp_counts.gt(1)].sum())
additional_timestamp_records = int(len(cleaned_frame) - timestamp_counts.size)
LOGGER.warning(
    "Preserved %d rows belonging to repeated timestamps (%d additional records) "
    "because they contain distinct weather observations",
    repeated_timestamp_rows,
    additional_timestamp_records,
)

2026-11-14 22:57:01,908 | WARNING | capstone.learning.data_cleaning | Dropped 17 exact duplicate rows; 48187 rows remain
2026-11-14 22:57:01,911 | WARNING | capstone.learning.data_cleaning | Preserved 13042 rows belonging to repeated timestamps (7612 additional records) because they contain distinct weather observations


## 8. Impute impossible values with month-specific medians

The assignment explicitly expects month-aware processing rather than one global average. The function below uses a loop over affected months. It falls back to the overall valid median only if a month contains no valid value. Every imputation is logged separately.

In [28]:
def impute_invalid_by_month(
    frame: pd.DataFrame,
    column: str,
    invalid_mask: pd.Series,
    reason: str,
) -> dict[str, dict[str, float | int]]:
    audit = {}
    month_key = frame["date_time"].dt.to_period("M")
    valid_global = frame.loc[~invalid_mask, column]
    global_median = float(valid_global.median())

    for month in sorted(month_key.loc[invalid_mask].unique()):
        target_mask = invalid_mask & month_key.eq(month)
        valid_month_mask = (~invalid_mask) & month_key.eq(month)
        monthly_values = frame.loc[valid_month_mask, column]
        if monthly_values.empty:
            replacement = global_median
            LOGGER.warning(
                "No valid monthly values for %s in %s; using global median %.4f",
                column,
                month,
                replacement,
            )
        else:
            replacement = float(monthly_values.median())

        affected_rows = int(target_mask.sum())
        frame.loc[target_mask, column] = replacement
        audit[str(month)] = {
            "affected_rows": affected_rows,
            "replacement_median": replacement,
        }
        LOGGER.warning(
            "Imputed %d %s values for %s in %s with monthly median %.4f: %s",
            affected_rows,
            column,
            column,
            month,
            replacement,
            reason,
        )
    return audit

temperature_invalid = cleaned_frame["temp"].le(0)
rain_invalid = cleaned_frame["rain_1h"].lt(0) | cleaned_frame["rain_1h"].gt(9000)

temperature_imputation_audit = impute_invalid_by_month(
    cleaned_frame,
    "temp",
    temperature_invalid,
    "temperature must be above 0 K",
)
rain_imputation_audit = impute_invalid_by_month(
    cleaned_frame,
    "rain_1h",
    rain_invalid,
    "rainfall exceeded the assignment's 9,000 mm plausibility limit",
)

display(
    {
        "temperature": temperature_imputation_audit,
        "rainfall": rain_imputation_audit,
    }
)

2026-11-14 22:57:01,919 | WARNING | capstone.learning.data_cleaning | Imputed 4 temp values for temp in 2014-01 with monthly median 261.2900: temperature must be above 0 K
2026-11-14 22:57:01,920 | WARNING | capstone.learning.data_cleaning | Imputed 6 temp values for temp in 2014-02 with monthly median 259.5000: temperature must be above 0 K
2026-11-14 22:57:01,922 | WARNING | capstone.learning.data_cleaning | Imputed 1 rain_1h values for rain_1h in 2016-07 with monthly median 0.0000: rainfall exceeded the assignment's 9,000 mm plausibility limit


{'temperature': {'2014-01': {'affected_rows': 4, 'replacement_median': 261.29},
  '2014-02': {'affected_rows': 6, 'replacement_median': 259.5}},
 'rainfall': {'2016-07': {'affected_rows': 1, 'replacement_median': 0.0}}}

## 9. Check other numerical boundaries without inventing changes

Snowfall cannot be negative, cloud percentage should be between 0 and 100, and traffic volume cannot be negative. The two zero-traffic rows are unusual but could represent a closure or sensor event, so they are flagged and retained until stronger evidence is available.

In [29]:
boundary_issues = {
    "negative_snow_rows": int(cleaned_frame["snow_1h"].lt(0).sum()),
    "clouds_outside_0_100_rows": int(
        (~cleaned_frame["clouds_all"].between(0, 100)).sum()
    ),
    "negative_traffic_rows": int(cleaned_frame["traffic_volume"].lt(0).sum()),
    "zero_traffic_rows_retained": int(cleaned_frame["traffic_volume"].eq(0).sum()),
}

if boundary_issues["zero_traffic_rows_retained"] > 0:
    LOGGER.warning(
        "Retained %d zero-traffic rows for investigation; no evidence supports imputation",
        boundary_issues["zero_traffic_rows_retained"],
    )
LOGGER.info("Completed numerical boundary checks: %s", boundary_issues)
display(pd.Series(boundary_issues, name="rows").to_frame())

2026-11-14 22:57:01,928 | WARNING | capstone.learning.data_cleaning | Retained 2 zero-traffic rows for investigation; no evidence supports imputation
2026-11-14 22:57:01,928 | INFO | capstone.learning.data_cleaning | Completed numerical boundary checks: {'negative_snow_rows': 0, 'clouds_outside_0_100_rows': 0, 'negative_traffic_rows': 0, 'zero_traffic_rows_retained': 2}


,rows
negative_snow_rows,0
clouds_outside_0_100_rows,0
negative_traffic_rows,0
zero_traffic_rows_retained,2


## 10. Validate the cleaned result

Validation is part of the pipeline, not a manual afterthought. A failed condition should stop the final script before it writes an invalid output.

In [30]:
cleaning_summary = {
    "raw_rows": len(raw_frame),
    "cleaned_rows": len(cleaned_frame),
    "rows_removed_as_duplicates": len(raw_frame) - len(cleaned_frame),
    "remaining_exact_duplicates": int(cleaned_frame.duplicated().sum()),
    "remaining_null_values": int(cleaned_frame.isna().sum().sum()),
    "remaining_nonpositive_temperature_rows": int(cleaned_frame["temp"].le(0).sum()),
    "remaining_extreme_rain_rows": int(cleaned_frame["rain_1h"].gt(9000).sum()),
    "unique_timestamps": int(cleaned_frame["date_time"].nunique()),
    "additional_timestamp_records_preserved": (
        len(cleaned_frame) - int(cleaned_frame["date_time"].nunique())
    ),
    "unique_weather_descriptions": int(
        cleaned_frame["weather_description"].nunique()
    ),
}

LOGGER.info("Cleaned dataset validation summary: %s", cleaning_summary)
display(pd.Series(cleaning_summary, name="value").to_frame())
display(cleaned_frame.head(3))

2026-11-14 22:57:01,945 | INFO | capstone.learning.data_cleaning | Cleaned dataset validation summary: {'raw_rows': 48204, 'cleaned_rows': 48187, 'rows_removed_as_duplicates': 17, 'remaining_exact_duplicates': 0, 'remaining_null_values': 0, 'remaining_nonpositive_temperature_rows': 0, 'remaining_extreme_rain_rows': 0, 'unique_timestamps': 40575, 'additional_timestamp_records_preserved': 7612, 'unique_weather_descriptions': 37}


,value
raw_rows,48204
cleaned_rows,48187
rows_removed_as_duplicates,17
remaining_exact_duplicates,0
remaining_null_values,0
remaining_nonpositive_temperature_rows,0
remaining_extreme_rain_rows,0
unique_timestamps,40575
additional_timestamp_records_preserved,7612
unique_weather_descriptions,37


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767


## 11. Run automated checks

In [31]:
assert raw_frame.shape == (48_204, 9)
assert initial_profile["exact_duplicate_rows"] == 17
assert initial_profile["zero_kelvin_rows"] == 10
assert initial_profile["rain_above_9000_rows"] == 1
assert category_change_counts["weather_description"] == 1_730
assert duplicate_count == 17
assert cleaning_summary["cleaned_rows"] == 48_187
assert cleaning_summary["rows_removed_as_duplicates"] == 17
assert cleaning_summary["remaining_exact_duplicates"] == 0
assert cleaning_summary["remaining_null_values"] == 0
assert cleaning_summary["remaining_nonpositive_temperature_rows"] == 0
assert cleaning_summary["remaining_extreme_rain_rows"] == 0
assert cleaning_summary["unique_timestamps"] == 40_575
assert cleaning_summary["additional_timestamp_records_preserved"] == 7_612
assert cleaning_summary["unique_weather_descriptions"] == 37
assert boundary_issues["zero_traffic_rows_retained"] == 2
assert pd.api.types.is_datetime64_any_dtype(cleaned_frame["date_time"])

print("PASS - Logged data-cleaning decisions matched the expected dataset.")

PASS - Logged data-cleaning decisions matched the expected dataset.


## 12. Interpretation checkpoint

Before creating `pipeline.py`, make sure you can explain:

1. Why schema validation happens before cleaning.
2. Why `None` in the holiday field is not treated as missing.
3. Why standardisation happens before duplicate removal.
4. Why repeated timestamps are preserved even after exact duplicates are removed.
5. Why the impossible values use monthly medians instead of a global average.
6. Why zero traffic is flagged but not automatically changed.
7. Which events use INFO, WARNING and ERROR logging levels.

### Stop point

Run all cells and confirm the final code cell reports `PASS`. Save the notebook. We will then promote this logic into the required production `pipeline.py`, add file logging, generate a cleaned CSV and create automated tests.